# Experiment 001 — Typo & ASR Robustness: GPU driver

Five cells: **setup → install → ASR pre-processing → generate → pull results**.
Runs on a single GPU (a 16 GB T4 works with `configs/fallback_t4.yaml` and the
small models). See `docs/PROVENANCE.md` for why each pinned version is used.


### Cell 1 — Setup: clone the repo and install the package

In [ ]:
# If running in Colab, clone your repository here. Locally, skip the clone.
# !git clone https://github.com/<your-org>/glamor-research-onboarding.git
# %cd glamor-research-onboarding/experiments/001_typo_robustness

import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=False)
print("package installed")


### Cell 2 — Install the GPU/ASR stack

Pinned for mutual compatibility (vLLM < 0.11, transformers < 5.0); confirm
against the machine's CUDA and bump together if needed (docs/PROVENANCE.md §8).

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt", "-q"], check=False)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Cell 3 — (once) Build the ASR perturbation set

TTS → (noise) → Whisper, deterministic. Caches audio under `data/audio/` and
writes `data/perturbations/asr_items.jsonl`. Skip if you only want the keyboard
arm for a first pass. Start small to validate, then scale `--reasoning-items`.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "tools/build_asr_items.py",
    "--output", "data/perturbations/asr_items.jsonl",
    "--audio-directory", "data/audio",
    "--reasoning-items", "40",          # scale up for the real run
], check=False)

### Cell 4 — Pin revisions, then generate

Fill in the model revisions before a confirmatory run (the cell prints each
current SHA). Then generate. Use `configs/fallback_t4.yaml` + a small model on a
T4; `configs/main.yaml` + `llama_8b_awq` on the cluster.

In [ ]:
# Resolve and record each roster model's commit SHA (pre-registration step).
# Requires HF auth for gated models (meta-llama/*, mistralai/*).
# Writes configs/pinned_revisions.yaml and prints copy-pasteable lines for
# src/inference/roster.py.
import subprocess, sys
subprocess.run([sys.executable, "tools/pin_revisions.py"], check=False)

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "tools/run_generation.py",
    "--config", "configs/fallback_t4.yaml",   # or configs/main.yaml
    "--model", "llama_1b",                     # or llama_8b_awq on the cluster
    "--backend", "vllm",
    "--output-directory", "results/colab",
], check=False)

### Cell 5 — Analyze and pull results

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "tools/run_analysis.py",
    "--generations", "results/colab/fallback_t4_generations.jsonl",
    "--output-directory", "analysis/colab",
], check=False)

# In Colab, download the analysis directory:
# from google.colab import files
# import shutil; shutil.make_archive("analysis_colab", "zip", "analysis/colab")
# files.download("analysis_colab.zip")
print("done; see analysis/colab/cell_table.csv and the figures")